# Day 20 - Plotly (Express + Graph Objects)

---

**What you will learn today:**

- Why Plotly exists and how it differs from Matplotlib and Seaborn
- Plotly Express (px): the fast, high-level API
- Graph Objects (go): the low-level API for full control
- All major chart types: scatter, line, bar, histogram, box, violin, pie, sunburst, treemap
- Geographic plots: choropleth maps, scatter_geo
- 3D plots: scatter3d, surface
- Animation: animation_frame parameter
- Customization: update_layout, update_traces, templates
- Subplots with make_subplots
- Saving as HTML and PNG

---

**Dataset today:** `px.data.gapminder()` - World development data from 1952 to 2007. Countries, continents, GDP per capita, life expectancy, population. Perfect for animated charts.

---

**Why this matters:**

Matplotlib and Seaborn produce static images. You save them as PNG and share them. The viewer cannot interact with them.

Plotly produces interactive HTML charts. The viewer can zoom in, pan, hover over individual data points to see exact values, click legend items to hide/show groups, and watch animations. This is the standard in modern data science dashboards and web applications.

A single `fig.write_html('dashboard.html')` gives you a file that works in any browser without Python. Upload it to GitHub Pages and you have a live portfolio piece.

---

# Part 1: Matplotlib vs Seaborn vs Plotly - Choosing the Right Tool

Understanding when to use which library saves you hours of frustration.

---

## Matplotlib

**What it is:** The foundational Python plotting library. Everything is built on top of it.

**Output:** Static PNG, PDF, SVG images.

**Strengths:**
- Maximum control over every pixel
- Works with raw NumPy arrays, no DataFrame needed
- 3D plots, custom animations, publication-quality figures
- Required for scientific papers (journals want PDF figures)

**Weaknesses:**
- Very verbose. 20 lines for what Seaborn does in 2.
- No interactivity (no hover, no zoom)
- Ugly defaults

**Use when:** Scientific papers, custom artistic charts, working with arrays not DataFrames.

---

## Seaborn

**What it is:** High-level statistical visualization built on Matplotlib.

**Output:** Static PNG images (same as Matplotlib).

**Strengths:**
- Beautiful defaults
- Statistical plots built in (KDE, confidence intervals, violin)
- hue, style, size encoding in one line
- Ideal for EDA during data analysis

**Weaknesses:**
- Still static, no interactivity
- Slower with large datasets
- Less control than Matplotlib

**Use when:** EDA in Jupyter notebooks, statistical analysis, academic presentations.

---

## Plotly

**What it is:** Interactive visualization library. Charts are HTML/JavaScript under the hood.

**Output:** Interactive HTML files that work in any browser. Also can export to PNG.

**Strengths:**
- Fully interactive: hover, zoom, pan, click to hide/show
- Animations with animation_frame parameter
- Geographic maps built in
- Save as HTML and share without Python
- Works in Dash (web app framework) and Streamlit
- 3D plots that you can rotate in browser

**Weaknesses:**
- Slower to render than Matplotlib/Seaborn
- Requires more memory for large datasets
- Overkill for quick exploratory analysis

**Use when:** Dashboards, portfolios, web apps, presentations where interactivity matters, sharing with non-technical stakeholders.

---

## px vs go - The Two APIs Within Plotly

Plotly has two layers:

**Plotly Express (px):** High-level, one-liner API. Like Seaborn but interactive.
```python
fig = px.scatter(df, x='gdpPercap', y='lifeExp', color='continent')
fig.show()
```

**Graph Objects (go):** Low-level API. Like Matplotlib. Full control.
```python
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['gdp'], y=df['life'], mode='markers'))
fig.show()
```

**Rule:** Start with px. When px does not give you enough control (subplots, mixed chart types, very custom layouts), switch to go.

In practice: px for quick charts, go for dashboards and production code. You can also mix them: create a px figure and then use go methods to customize it.

In [2]:
# Install plotly if needed (already installed in Colab)
# !pip install plotly --quiet

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os

os.makedirs('day20_charts', exist_ok=True)

# Load Gapminder dataset
gapminder = px.data.gapminder()

print('Plotly version:', px.__version__ if hasattr(px, '__version__') else 'loaded')
print()
print('Gapminder dataset:')
print(gapminder.head())
print(f'\nShape: {gapminder.shape}')
print(f'Years: {sorted(gapminder["year"].unique())}')
print(f'Continents: {gapminder["continent"].unique()}')
print(f'Countries: {gapminder["country"].nunique()}')

Plotly version: loaded

Gapminder dataset:
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  

Shape: (1704, 8)
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
Countries: 142


### Understanding the Gapminder Dataset

Columns:
- `country` - country name (categorical)
- `continent` - continent name (categorical, 5 values)
- `year` - year of measurement: 1952, 1957, 1962, ..., 2007 (every 5 years)
- `lifeExp` - life expectancy at birth in years (numerical)
- `pop` - population (numerical, ranges from ~60,000 to 1.3 billion)
- `gdpPercap` - GDP per capita in USD (numerical)
- `iso_alpha` - 3-letter country code (used for maps)
- `iso_num` - numeric country code

This dataset is perfect for animated charts because it has a time dimension (year). You can literally watch the world develop from 1952 to 2007.

---

# Part 2: Plotly Express Scatter Plot

## The Animated Bubble Chart - The Most Impressive One-Liner in Data Science

The animated scatter plot (bubble chart) using Gapminder is famous in data visualization. Hans Rosling, a Swedish statistician, made this chart famous in a TED talk. He showed how the world changed from 1952 to 2007 - countries getting richer and living longer.

With Plotly, you can recreate this entire animation in one line of code.

### How animation_frame Works:

When you set `animation_frame='year'`, Plotly creates one "frame" of the chart for each unique value of year. The Play button at the bottom cycles through the frames. Each frame is a complete snapshot of the data for that year.

### Encoding 5 Variables in One Chart:

- x-axis = GDP per capita (wealth)
- y-axis = life expectancy (health)
- bubble size = population
- color = continent
- animation frame = year (time)
- hover = country name

Six dimensions of information in one chart. This is what visualization is for.

In [3]:
# THE FAMOUS ANIMATED BUBBLE CHART - one line of code

fig = px.scatter(
    gapminder,
    x='gdpPercap',              # x-axis: GDP per capita
    y='lifeExp',                # y-axis: life expectancy
    size='pop',                 # bubble size: population
    color='continent',          # bubble color: continent
    hover_name='country',       # shown in BOLD at top of hover box
    hover_data={                # additional data shown in hover box
        'pop': ':,.0f',         # format: comma-separated integer
        'gdpPercap': ':,.0f',
        'year': True
    },
    animation_frame='year',     # creates animation slider + play button
    animation_group='country',  # tracks same country across frames (smooth animation)
    size_max=60,                # maximum bubble size in pixels
    log_x=True,                 # log scale on x-axis (GDP spans orders of magnitude)
    range_x=[200, 100000],      # fixed x range (so axis does not rescale during animation)
    range_y=[25, 90],           # fixed y range
    color_discrete_sequence=px.colors.qualitative.Set1,
    title='World Development: GDP vs Life Expectancy (1952-2007)',
    labels={
        'gdpPercap': 'GDP per Capita (USD, log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'pop': 'Population',
        'continent': 'Continent'
    }
)

# update_layout: modifies the overall figure layout
fig.update_layout(
    width=900,
    height=600,
    template='plotly_white',    # clean white background
    font=dict(size=13)
)

# Save as HTML - this file works in any browser without Python
fig.write_html('day20_charts/1_animated_bubble.html')
fig.show()
print('Animated bubble chart saved as HTML')
print('Click the Play button to watch the animation')
print('Hover over any bubble to see country details')

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# SCATTER - Static (no animation) with facets
# facet_col splits into separate charts by continent

gapminder_2007 = gapminder[gapminder['year'] == 2007]

fig = px.scatter(
    gapminder_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    color='continent',
    hover_name='country',
    facet_col='continent',      # one chart per continent, arranged in columns
    facet_col_wrap=3,           # max 3 per row, then wrap
    size_max=40,
    log_x=True,
    title='GDP vs Life Expectancy by Continent (2007)',
    labels={'gdpPercap': 'GDP per Capita (log)', 'lifeExp': 'Life Expectancy'}
)

fig.update_layout(width=1000, height=600, showlegend=False, template='plotly_white')
fig.write_html('day20_charts/2_scatter_facet.html')
fig.show()
print('Faceted scatter saved')

### Key Scatter Parameters:

- `x=`, `y=` - column names for axes
- `color=` - column to color by (like hue in Seaborn)
- `size=` - column for bubble size
- `symbol=` - column for marker shape (like style in Seaborn)
- `hover_name=` - column shown in bold at top of hover tooltip
- `hover_data=` - dict of additional columns to show in hover
- `animation_frame=` - column to animate over (creates Play button)
- `animation_group=` - column that identifies same entity across frames
- `facet_col=` - split into columns by this column
- `facet_row=` - split into rows
- `facet_col_wrap=` - max columns before wrapping
- `log_x=True` - logarithmic x-axis
- `log_y=True` - logarithmic y-axis
- `range_x=`, `range_y=` - fix the axis range (critical for animations so axes do not jump)
- `size_max=` - maximum bubble size in pixels
- `color_discrete_sequence=` - list of colors for categorical data
- `color_continuous_scale=` - colormap name for numerical data

---

# Part 3: Line Plot

## When to Use Line Plot in Plotly

Same rule as Matplotlib: when x-axis is time or a continuous sequence and you want to show trends.

The advantage over Matplotlib: hovering shows exact values. For time series with many lines (like multiple countries), you can click a legend item to hide that line. This is impossible with static charts.

In [ ]:
# LINE PLOT: Life expectancy over time for selected countries

selected_countries = ['India', 'China', 'United States', 'Brazil', 'Nigeria', 'Germany']
df_countries = gapminder[gapminder['country'].isin(selected_countries)]

fig = px.line(
    df_countries,
    x='year',
    y='lifeExp',
    color='country',             # one line per country
    markers=True,                # show dots at each data point
    line_dash='country',         # different dash style per country (solid, dashed, dotted)
    hover_data={'gdpPercap': ':,.0f', 'pop': ':,.0f'},
    title='Life Expectancy Trends: Selected Countries (1952-2007)',
    labels={'lifeExp': 'Life Expectancy (years)', 'year': 'Year', 'country': 'Country'}
)

fig.update_traces(
    line=dict(width=2.5),        # line thickness
    marker=dict(size=7)          # dot size
)

fig.update_layout(
    width=900, height=500,
    template='plotly_white',
    hovermode='x unified',       # shows all country values for the same x when hovering
    legend=dict(title='Country', yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.write_html('day20_charts/3_line_countries.html')
fig.show()
print('hovermode=x unified: hover anywhere on x to see all country values simultaneously')

---

# Part 4: Bar Chart

## barmode: group vs stack vs overlay vs relative

When you have multiple bars per x category (from color=), `barmode` controls how they are arranged:

- `barmode='group'` - bars side by side. Best for comparing values between groups.
- `barmode='stack'` - bars stacked. Best for showing total and composition simultaneously.
- `barmode='relative'` - stacked but negative values go down, positive go up.
- `barmode='overlay'` - bars overlap (needs opacity to see). Rarely used.

The `text_auto=True` parameter automatically adds value labels on top of (or inside) each bar.

In [ ]:
# BAR CHART 1: Top 10 countries by population in 2007

top10 = (gapminder_2007
         .nlargest(10, 'pop')
         .sort_values('pop', ascending=True))  # ascending for horizontal: largest at top

fig = px.bar(
    top10,
    x='pop',
    y='country',
    color='continent',
    orientation='h',             # 'h' = horizontal bars, 'v' = vertical (default)
    text_auto='.2s',             # auto text: .2s = 2 significant figures with SI suffix (B, M, K)
    hover_data={'gdpPercap': ':,.0f', 'lifeExp': ':.1f'},
    title='Top 10 Countries by Population (2007)',
    labels={'pop': 'Population', 'country': 'Country', 'continent': 'Continent'}
)

fig.update_layout(
    width=800, height=500,
    template='plotly_white',
    xaxis_title='Population',
    yaxis_title='',
    showlegend=True
)

fig.write_html('day20_charts/4_bar_top10.html')
fig.show()

In [ ]:
# BAR CHART 2: Grouped and stacked comparison
# Average life expectancy by continent over selected years

years_selected = [1952, 1972, 1992, 2007]
df_grouped = (gapminder[gapminder['year'].isin(years_selected)]
              .groupby(['continent', 'year'])['lifeExp']
              .mean()
              .reset_index()
              .rename(columns={'lifeExp': 'avg_lifeExp'}))

df_grouped['year'] = df_grouped['year'].astype(str)  # treat year as category for grouping

fig_group = px.bar(
    df_grouped,
    x='continent',
    y='avg_lifeExp',
    color='year',
    barmode='group',             # side by side
    text_auto='.1f',
    title='Average Life Expectancy by Continent (barmode=group)',
    labels={'avg_lifeExp': 'Avg Life Expectancy', 'continent': 'Continent', 'year': 'Year'}
)
fig_group.update_layout(width=900, height=450, template='plotly_white')
fig_group.write_html('day20_charts/4b_bar_group.html')
fig_group.show()

fig_stack = px.bar(
    df_grouped,
    x='continent',
    y='avg_lifeExp',
    color='year',
    barmode='stack',             # stacked
    title='Average Life Expectancy by Continent (barmode=stack)',
    labels={'avg_lifeExp': 'Avg Life Expectancy (stacked)', 'continent': 'Continent'}
)
fig_stack.update_layout(width=900, height=450, template='plotly_white')
fig_stack.write_html('day20_charts/4c_bar_stack.html')
fig_stack.show()

---

# Part 5: Histogram

## The marginal Parameter - Plotly's Unique Feature

Plotly histograms have a `marginal=` parameter that does not exist in Matplotlib or Seaborn. It adds a secondary plot ABOVE the histogram showing the same data in a different form.

- `marginal='box'` - adds a box plot above the histogram
- `marginal='violin'` - adds a violin plot above
- `marginal='rug'` - adds a rug plot (individual tick marks)

This gives you two views of the distribution in one chart. The box plot above shows the median and outliers while the histogram below shows the shape. Both are interactive and linked.

In [ ]:
# HISTOGRAM with marginal='box'

fig = px.histogram(
    gapminder_2007,
    x='lifeExp',
    nbins=30,                    # number of bins
    color='continent',           # overlapping histograms per continent (like hue)
    barmode='overlay',           # overlapping (not stacked)
    opacity=0.6,                 # transparency for overlapping
    marginal='box',              # box plot above the histogram
    hover_data=['country'],      # show country in hover
    title='Life Expectancy Distribution by Continent (2007)',
    labels={'lifeExp': 'Life Expectancy (years)', 'continent': 'Continent'}
)

fig.update_layout(
    width=900, height=500,
    template='plotly_white',
    bargap=0.1                   # gap between bars (0=no gap, 1=all gap)
)

fig.write_html('day20_charts/5_histogram_marginal.html')
fig.show()
print('marginal=box adds a box plot above the histogram - both are interactive')

In [ ]:
# HISTOGRAM comparison: all marginal types

india_data = gapminder[gapminder['country'] == 'India']

for marginal_type in ['box', 'violin', 'rug']:
    fig = px.histogram(
        gapminder_2007,
        x='gdpPercap',
        nbins=25,
        color='continent',
        barmode='overlay',
        opacity=0.5,
        marginal=marginal_type,
        log_x=True,
        title=f'GDP per Capita Distribution (marginal={marginal_type})',
        labels={'gdpPercap': 'GDP per Capita (log scale)'}
    )
    fig.update_layout(width=800, height=450, template='plotly_white')
    fig.write_html(f'day20_charts/5b_hist_marginal_{marginal_type}.html')

print('All three marginal types saved')
fig.show()  # shows the last one (rug)

---

# Part 6: Box Plot and Violin Plot

## Plotly Box Plot Advantages Over Seaborn:

- Hover over any outlier dot to see exactly which country it is
- `points='all'` shows all individual data points alongside the box
- `notched=True` adds notches to the box: if two boxes' notches do not overlap, the medians are statistically significantly different
- Click legend to hide/show specific continents

In [ ]:
# BOX PLOT: GDP per capita by continent

fig = px.box(
    gapminder_2007,
    x='continent',
    y='gdpPercap',
    color='continent',
    points='all',                # 'all' = show all points, 'outliers' = only outliers, False = none
    notched=True,                # notch at median - overlapping notches = not significantly different
    hover_name='country',        # country name in bold on hover
    log_y=True,                  # log scale because GDP varies enormously
    title='GDP per Capita by Continent (2007) - points=all, notched=True',
    labels={'gdpPercap': 'GDP per Capita (USD, log)', 'continent': 'Continent'}
)

fig.update_layout(
    width=900, height=500,
    template='plotly_white',
    showlegend=False             # legend redundant when x=continent and color=continent
)

fig.write_html('day20_charts/6_box.html')
fig.show()
print('Hover over any dot to see the country name and exact GDP')

In [ ]:
# VIOLIN PLOT: box=True shows a mini box plot inside the violin

fig = px.violin(
    gapminder_2007,
    x='continent',
    y='lifeExp',
    color='continent',
    box=True,                    # mini boxplot inside the violin
    points='all',                # show all individual points
    hover_name='country',
    title='Life Expectancy Distribution by Continent (2007) - Violin with box=True',
    labels={'lifeExp': 'Life Expectancy (years)', 'continent': 'Continent'}
)

fig.update_layout(
    width=900, height=500,
    template='plotly_white',
    showlegend=False
)

fig.write_html('day20_charts/7_violin.html')
fig.show()

---

# Part 7: Pie Chart and Donut Chart

## The hole Parameter - Converting Pie to Donut

`hole=` takes a value from 0 to 1. It is the radius of the hole as a fraction of the chart radius.
- `hole=0` - regular pie chart
- `hole=0.4` - donut chart with 40% hole radius
- `hole=0.7` - thin donut ring

Donut charts are often preferred in professional dashboards because you can put a number or label in the center hole, and they look more modern.

In [ ]:
# PIE and DONUT charts

continent_pop = (gapminder_2007
                 .groupby('continent')['pop']
                 .sum()
                 .reset_index()
                 .rename(columns={'pop': 'total_population'}))

print('World population by continent (2007):')
print(continent_pop)

# Pie chart
fig_pie = px.pie(
    continent_pop,
    names='continent',
    values='total_population',
    title='World Population by Continent (2007)',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig_pie.update_traces(
    textposition='inside',       # where to put the percentage label
    textinfo='percent+label',    # show both percent and label
    hovertemplate='%{label}<br>Population: %{value:,.0f}<br>Share: %{percent}'
)
fig_pie.update_layout(width=600, height=500)
fig_pie.write_html('day20_charts/8_pie.html')
fig_pie.show()

# Donut chart - same data, just add hole=
fig_donut = px.pie(
    continent_pop,
    names='continent',
    values='total_population',
    hole=0.45,                   # THIS converts pie to donut
    title='World Population by Continent - Donut Chart (hole=0.45)',
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig_donut.update_traces(textposition='outside', textinfo='percent+label')
fig_donut.update_layout(width=600, height=500)
fig_donut.write_html('day20_charts/8b_donut.html')
fig_donut.show()

---

# Part 8: Sunburst and Treemap - Hierarchical Data

## When Data has Hierarchy

Sometimes your data has nested categories. World population is divided into continents, and each continent is divided into countries. This is a two-level hierarchy.

Pie charts only show one level. Sunburst and Treemap show the entire hierarchy at once.

- **Sunburst:** Concentric rings. Inner ring = top level (continent). Outer ring = next level (country). Click a continent to zoom in and see just its countries.

- **Treemap:** Nested rectangles. Outer rectangle = continent. Inner smaller rectangles = countries within that continent. Size = population.

Both are fully interactive in Plotly. You can click to zoom into any section.

### The path= Parameter:

`path=[column1, column2]` defines the hierarchy from outer to inner. `path=['continent', 'country']` means continent is the first level and country is the second.

In [ ]:
# SUNBURST: World population hierarchy

fig = px.sunburst(
    gapminder_2007,
    path=['continent', 'country'],  # hierarchy: continent -> country
    values='pop',                    # size of each segment = population
    color='lifeExp',                 # color by life expectancy
    color_continuous_scale='RdYlGn',  # red (low) to green (high)
    hover_data={'gdpPercap': ':,.0f'},
    title='World Population: Continent -> Country (color=life expectancy)',
    maxdepth=2                       # show up to 2 levels deep initially
)

fig.update_layout(
    width=700, height=700,
    coloraxis_colorbar=dict(title='Life Expectancy')
)

fig.write_html('day20_charts/9_sunburst.html')
fig.show()
print('Click any continent segment to zoom in and see individual countries')

In [ ]:
# TREEMAP: Same data, different visual encoding

fig = px.treemap(
    gapminder_2007,
    path=['continent', 'country'],
    values='pop',
    color='gdpPercap',
    color_continuous_scale='Blues',
    hover_data={'lifeExp': ':.1f', 'pop': ':,.0f'},
    title='World Population Treemap: Size=Population, Color=GDP per Capita'
)

fig.update_traces(
    textinfo='label+percent parent',  # show name and % of parent
    hovertemplate='<b>%{label}</b><br>Population: %{value:,.0f}<br>GDP: %{color:,.0f}'
)

fig.update_layout(
    width=900, height=600,
    coloraxis_colorbar=dict(title='GDP per Capita')
)

fig.write_html('day20_charts/10_treemap.html')
fig.show()
print('Click any continent to zoom into its countries')

---

# Part 9: Geographic Maps

## Two Types of Geographic Charts in Plotly

**Choropleth map:** Colors an entire country/region based on a value. Like a heatmap on a world map. Perfect for showing which countries have the highest GDP, population, etc.

**Scatter_geo:** Places dots at geographic coordinates (latitude/longitude). Dot size and color can encode additional variables. Perfect for showing specific locations (cities, events, facilities).

### locationmode in Choropleth:

- `locationmode='ISO-3'` - use 3-letter country codes (e.g., 'IND', 'USA'). Most reliable.
- `locationmode='country names'` - use full country names. Can fail if spelling does not match.
- `locationmode='USA-states'` - for US state-level data (2-letter codes like 'CA', 'TX')

The Gapminder dataset has `iso_alpha` column which contains ISO-3 codes. This is why it works perfectly for choropleth maps.

In [ ]:
# CHOROPLETH MAP with animation
# This is arguably the most impressive chart you can make in one line

fig = px.choropleth(
    gapminder,
    locations='iso_alpha',           # column with ISO-3 country codes
    locationmode='ISO-3',            # tells Plotly: iso_alpha contains ISO-3 codes
    color='gdpPercap',               # country fill color = GDP
    hover_name='country',
    hover_data={'lifeExp': ':.1f', 'pop': ':,.0f', 'gdpPercap': ':,.0f'},
    animation_frame='year',          # animate over time
    color_continuous_scale='Viridis',# colormap
    range_color=[0, 50000],          # fix color range across all frames
    title='World GDP per Capita (1952-2007) - Animated Choropleth',
    labels={'gdpPercap': 'GDP per Capita (USD)'}
)

fig.update_layout(
    width=1000, height=550,
    geo=dict(
        showframe=False,             # no frame around the map
        showcoastlines=True,
        coastlinecolor='white',
        projection_type='natural earth'  # map projection style
    ),
    coloraxis_colorbar=dict(title='GDP per Capita')
)

fig.write_html('day20_charts/11_choropleth_animated.html')
fig.show()
print('Click Play to watch GDP develop country by country from 1952 to 2007')

In [ ]:
# SCATTER_GEO: Bubble map on world map
# Each country is a bubble at its geographic location
# Size = population, Color = continent

fig = px.scatter_geo(
    gapminder_2007,
    locations='iso_alpha',
    size='pop',                      # bubble size = population
    color='continent',               # bubble color = continent
    hover_name='country',
    hover_data={'gdpPercap': ':,.0f', 'lifeExp': ':.1f'},
    size_max=50,
    projection='natural earth',
    title='World Population by Country (2007) - Bubble Map',
    labels={'pop': 'Population'}
)

fig.update_layout(
    width=1000, height=500,
    geo=dict(showframe=False, showcoastlines=True, coastlinecolor='lightgray',
             showland=True, landcolor='lightyellow', showocean=True, oceancolor='lightcyan')
)

fig.write_html('day20_charts/12_scatter_geo.html')
fig.show()

---

# Part 10: 3D Plots

## 3D Scatter

Plotly 3D charts are fully interactive - you can rotate them with mouse drag, zoom with scroll wheel, and hover to see values. This is impossible with Matplotlib 3D which is a static image.

## 3D Surface with Meshgrid

For surface plots, you need Graph Objects (go) because px does not have a surface chart. The workflow is identical to Matplotlib Day 17: create meshgrid, compute Z, pass to go.Surface.

In [ ]:
# 3D SCATTER: GDP, Life Expectancy, Population in 3D

fig = px.scatter_3d(
    gapminder_2007,
    x='gdpPercap',
    y='lifeExp',
    z='pop',                         # third axis
    color='continent',
    size='pop',
    size_max=30,
    hover_name='country',
    log_x=True,
    log_z=True,                      # log scale on z-axis
    title='3D: GDP vs Life Expectancy vs Population (2007)',
    labels={
        'gdpPercap': 'GDP (log)',
        'lifeExp': 'Life Exp',
        'pop': 'Population (log)'
    },
    opacity=0.8
)

fig.update_layout(
    width=800, height=600,
    scene=dict(
        xaxis_title='GDP per Capita (log)',
        yaxis_title='Life Expectancy',
        zaxis_title='Population (log)'
    )
)

fig.write_html('day20_charts/13_3d_scatter.html')
fig.show()
print('Drag to rotate. Scroll to zoom. Hover for country details.')

In [ ]:
# 3D SURFACE PLOT with Graph Objects
# go.Surface needs meshgrid format - same as Matplotlib Day 17

x_range = np.linspace(-4, 4, 80)
y_range = np.linspace(-4, 4, 80)
XX, YY = np.meshgrid(x_range, y_range)
R = np.sqrt(XX**2 + YY**2)
ZZ = np.sin(R) / (R + 1e-6)  # sinc function: circular ripples

fig = go.Figure(data=[
    go.Surface(
        x=XX,
        y=YY,
        z=ZZ,
        colorscale='Plasma',     # colormap name
        showscale=True,          # show colorbar
        opacity=0.9,
        hovertemplate='x: %{x:.2f}<br>y: %{y:.2f}<br>z: %{z:.3f}<extra></extra>'
    )
])

fig.update_layout(
    title='3D Surface Plot: sin(r)/r (Interactive - Drag to Rotate)',
    width=800, height=600,
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z = sin(r)/r',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.0)  # initial camera position
        )
    ),
    template='plotly_dark'       # dark theme
)

fig.write_html('day20_charts/14_3d_surface.html')
fig.show()
print('Drag with mouse to rotate. Zoom with scroll.')

---

# Part 11: Funnel Chart

## What is a Funnel Chart?

A funnel chart shows a process where a large number of things enter at the top and progressively fewer make it through each stage.

Classic use cases:
- Sales pipeline: 10,000 website visitors -> 2,000 sign-ups -> 500 trial starts -> 100 purchases
- Hiring pipeline: 500 applications -> 200 screenings -> 50 interviews -> 10 offers -> 3 hires
- User onboarding: 1000 app downloads -> 700 opened app -> 400 completed tutorial -> 200 active users

The width of each stage is proportional to the count. You can immediately see where the biggest drop-offs happen.

In [ ]:
# FUNNEL CHART: Sales pipeline example

funnel_data = pd.DataFrame({
    'stage': ['Website Visitors', 'Product Page Views', 'Add to Cart', 'Checkout Started', 'Purchase Completed'],
    'count': [50000, 15000, 5000, 2000, 800],
    'conversion': ['100%', '30%', '10%', '4%', '1.6%']
})

fig = px.funnel(
    funnel_data,
    x='count',
    y='stage',
    color='stage',
    title='E-Commerce Sales Funnel: Website Visitors to Purchase',
    labels={'count': 'Number of Users', 'stage': 'Funnel Stage'}
)

fig.update_layout(
    width=700, height=450,
    template='plotly_white',
    showlegend=False
)

fig.write_html('day20_charts/15_funnel.html')
fig.show()

---

# Part 12: Graph Objects (go) - The Low-Level API

## When to Use go Instead of px

px is great for standard charts. But for:
- Multiple chart types in one figure (subplots with different chart types)
- Very precise control over trace appearance
- Adding multiple traces to one chart
- Custom hover templates
- Things px simply does not support

You need go.

## The go Workflow

```python
fig = go.Figure()                    # creates empty figure
fig.add_trace(go.Scatter(...))       # adds a scatter trace
fig.add_trace(go.Bar(...))           # adds a bar trace
fig.update_layout(...)               # modifies figure settings
fig.update_traces(...)               # modifies all traces at once
fig.show()
```

A **trace** is one dataset plotted on the chart. One chart can have multiple traces. For example, a line chart with 5 countries has 5 traces - one per country.

## mode parameter in go.Scatter

- `mode='lines'` - only lines
- `mode='markers'` - only dots
- `mode='lines+markers'` - lines with dots
- `mode='text'` - only text labels
- `mode='lines+markers+text'` - all three

In [ ]:
# GRAPH OBJECTS: Multiple traces on one chart
# Comparing life expectancy of India, China, USA over time

countries_data = {
    'India':         gapminder[gapminder['country'] == 'India'],
    'China':         gapminder[gapminder['country'] == 'China'],
    'United States': gapminder[gapminder['country'] == 'United States']
}

colors = {'India': 'coral', 'China': 'steelblue', 'United States': 'green'}

fig = go.Figure()

for country, df_c in countries_data.items():
    fig.add_trace(
        go.Scatter(
            x=df_c['year'],
            y=df_c['lifeExp'],
            mode='lines+markers',       # line with dots
            name=country,               # legend label
            line=dict(
                color=colors[country],
                width=2.5
            ),
            marker=dict(
                size=8,
                symbol='circle'
            ),
            # custom hover template:
            # %{x} = x value, %{y} = y value
            # <extra></extra> removes the trace name box from hover
            hovertemplate=f'<b>{country}</b><br>Year: %{{x}}<br>Life Expectancy: %{{y:.1f}} years<extra></extra>'
        )
    )

# update_layout sets overall figure properties
fig.update_layout(
    title='Life Expectancy: India, China, USA (1952-2007)',
    xaxis_title='Year',
    yaxis_title='Life Expectancy (years)',
    template='plotly_white',
    width=900, height=500,
    legend=dict(title='Country', yanchor='top', y=0.99, xanchor='left', x=0.01),
    hovermode='x unified'
)

fig.write_html('day20_charts/16_go_scatter.html')
fig.show()

In [ ]:
# GRAPH OBJECTS: go.Bar, go.Histogram, go.Box, go.Pie examples

# go.Bar
top5_2007 = gapminder_2007.nlargest(5, 'pop')

fig_bar = go.Figure()
fig_bar.add_trace(
    go.Bar(
        x=top5_2007['country'],
        y=top5_2007['pop'],
        marker_color=['steelblue', 'coral', 'green', 'orange', 'purple'],
        text=[f'{p/1e9:.2f}B' for p in top5_2007['pop']],  # custom labels
        textposition='outside',
        hovertemplate='%{x}<br>Population: %{y:,.0f}<extra></extra>'
    )
)
fig_bar.update_layout(
    title='Top 5 Countries by Population (go.Bar)',
    xaxis_title='Country',
    yaxis_title='Population',
    template='plotly_white',
    width=700, height=400
)
fig_bar.write_html('day20_charts/17_go_bar.html')
fig_bar.show()

In [ ]:
# update_layout vs update_traces - know the difference

# update_layout: modifies the FIGURE (axes, title, legend, size, template)
# update_traces: modifies ALL TRACES (marker properties, line properties, opacity)

fig = px.scatter(gapminder_2007, x='gdpPercap', y='lifeExp', color='continent', log_x=True)

# Modify the figure layout
fig.update_layout(
    title=dict(text='update_layout and update_traces Demo', font=dict(size=16)),
    xaxis_title='GDP per Capita (log scale)',
    yaxis_title='Life Expectancy',
    template='plotly_dark',      # dark background
    width=800, height=500,
    showlegend=True,
    legend=dict(
        title='Continent',
        bgcolor='rgba(0,0,0,0.5)',  # semi-transparent background
        bordercolor='white',
        borderwidth=1
    )
)

# Modify all traces at once
fig.update_traces(
    marker=dict(
        size=10,
        opacity=0.8,
        line=dict(width=1, color='white')  # white border around each marker
    )
)

fig.show()
print('update_layout = figure-level settings')
print('update_traces = trace-level settings (applies to all traces)')

---

# Part 13: Subplots with make_subplots

## How make_subplots Works

`make_subplots` creates a figure with a grid of subplot areas. Then you add traces to specific positions using `row=` and `col=` parameters.

This is the go equivalent of `plt.subplots()` in Matplotlib, but for interactive charts.

### Key Parameters of make_subplots:

- `rows=`, `cols=` - grid dimensions
- `subplot_titles=` - list of titles for each subplot
- `shared_xaxes=True` - all subplots share the same x-axis
- `shared_yaxes=True` - all subplots share the same y-axis
- `specs=` - 2D list specifying type for each cell (`{'type': 'pie'}`, `{'type': 'scene'}` for 3D)
- `horizontal_spacing=` - gap between columns (0 to 1)
- `vertical_spacing=` - gap between rows (0 to 1)

In [ ]:
# make_subplots: 2x2 dashboard

np.random.seed(42)
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
sales  = [120, 135, 110, 145, 160, 155, 170, 180, 165, 190, 210, 250]
profit = [18, 22, 15, 28, 35, 30, 40, 45, 38, 50, 62, 80]
product_categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Sports']
category_sales = [45000, 30000, 25000, 15000, 20000]

# Create 2x2 grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Monthly Sales (Line + Markers)',
        'Monthly Profit (Bar)',
        'Sales Distribution (Box)',
        'Category Sales Share (Pie)'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],    # row 1: scatter and bar
        [{'type': 'box'},     {'type': 'pie'}]      # row 2: box and pie
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# Top-left: Line chart (go.Scatter)
fig.add_trace(
    go.Scatter(
        x=months, y=sales,
        mode='lines+markers',
        name='Sales',
        line=dict(color='steelblue', width=2.5),
        marker=dict(size=8),
        hovertemplate='%{x}: %{y} units<extra></extra>'
    ),
    row=1, col=1
)

# Top-right: Bar chart (go.Bar)
fig.add_trace(
    go.Bar(
        x=months, y=profit,
        name='Profit',
        marker_color='coral',
        text=[f'{p}K' for p in profit],
        textposition='outside',
        hovertemplate='%{x}: %{y}K profit<extra></extra>'
    ),
    row=1, col=2
)

# Bottom-left: Box plot (go.Box)
# Showing distribution of sales across all months
fig.add_trace(
    go.Box(
        y=sales,
        name='Sales Distribution',
        marker_color='mediumpurple',
        boxpoints='all',          # show all individual points
        jitter=0.3,               # horizontal jitter
        pointpos=-1.5             # position of points relative to box
    ),
    row=2, col=1
)

# Bottom-right: Pie chart (go.Pie)
fig.add_trace(
    go.Pie(
        labels=product_categories,
        values=category_sales,
        name='Category Share',
        hole=0.35,                # donut style
        textinfo='percent+label',
        marker_colors=px.colors.qualitative.Set2
    ),
    row=2, col=2
)

# Overall figure layout
fig.update_layout(
    title=dict(
        text='Sales Dashboard - make_subplots 2x2 Grid',
        font=dict(size=18)
    ),
    width=1100, height=700,
    template='plotly_dark',      # dark theme - portfolio-ready
    showlegend=False,
    paper_bgcolor='#1a1a2e',     # dark navy background
    plot_bgcolor='#16213e'
)

# Update axis labels for specific subplots
fig.update_xaxes(title_text='Month', row=1, col=1)
fig.update_yaxes(title_text='Units Sold', row=1, col=1)
fig.update_xaxes(title_text='Month', row=1, col=2)
fig.update_yaxes(title_text='Profit (thousands)', row=1, col=2)

fig.write_html('day20_charts/18_dashboard_2x2.html')
fig.show()
print('2x2 Dashboard saved as HTML - open in browser for full interactivity')

---

# Part 14: Saving and Exporting

## write_html - For Sharing and Portfolios

`fig.write_html('file.html')` saves the chart as a standalone HTML file. This file:
- Works in any browser (Chrome, Firefox, Safari, Edge)
- Does not require Python or any installation
- Has full interactivity (zoom, hover, pan, click legend)
- Can be uploaded to GitHub and viewed via GitHub Pages
- Can be sent via email - recipient just opens it

Parameters:
- `include_plotlyjs=True` - includes the full Plotly JS library in the file (~3MB). File is self-contained.
- `include_plotlyjs='cdn'` - links to the online CDN. File is tiny (~5KB) but needs internet.
- `full_html=True` - complete HTML page with head/body tags
- `full_html=False` - just the div. Useful for embedding in existing web pages.

## write_image - For Static Export

`fig.write_image('file.png')` saves as PNG (requires kaleido package).
Also supports: `.jpeg`, `.svg`, `.pdf`, `.webp`

Parameters: `width=`, `height=`, `scale=` (scale=2 doubles resolution)

In [ ]:
# Saving examples

fig_save = px.scatter(gapminder_2007, x='gdpPercap', y='lifeExp',
                      color='continent', size='pop', hover_name='country',
                      log_x=True, title='Save Demo')

# Save as self-contained HTML
fig_save.write_html(
    'day20_charts/save_demo.html',
    include_plotlyjs=True,      # self-contained (works offline)
    full_html=True
)
print('HTML saved (self-contained, works offline)')

# Save as HTML with CDN (much smaller file)
fig_save.write_html(
    'day20_charts/save_demo_cdn.html',
    include_plotlyjs='cdn',     # loads Plotly from internet (needs connection)
    full_html=True
)
print('HTML saved (CDN version, needs internet, much smaller file)')

# Save as PNG (requires kaleido)
try:
    fig_save.write_image('day20_charts/save_demo.png', width=900, height=500, scale=2)
    print('PNG saved (static image, high resolution)')
except Exception as e:
    print(f'PNG save needs kaleido: pip install kaleido')
    print(f'Error: {e}')

---

# GitHub File 1: plotly_express.py - Gapminder Dashboard

In [ ]:
# ============================================================
# day20_plotly/plotly_express.py
# Gapminder Dataset - 4 Plotly Express Charts
# ============================================================

import plotly.express as px
import pandas as pd
import os

os.makedirs('day20_charts', exist_ok=True)

gapminder = px.data.gapminder()
gapminder_2007 = gapminder[gapminder['year'] == 2007]

print(f'Gapminder: {gapminder.shape} rows, {gapminder["country"].nunique()} countries')

# -------------------------------------------------------
# CHART 1: Animated Bubble Chart - GDP vs Life Expectancy
# -------------------------------------------------------
fig1 = px.scatter(
    gapminder,
    x='gdpPercap', y='lifeExp',
    size='pop', color='continent',
    hover_name='country',
    hover_data={'pop': ':,.0f', 'gdpPercap': ':,.0f'},
    animation_frame='year',
    animation_group='country',
    size_max=60, log_x=True,
    range_x=[200, 100000], range_y=[25, 90],
    title='World Development: GDP vs Life Expectancy (1952-2007)',
    labels={'gdpPercap': 'GDP per Capita (USD, log)', 'lifeExp': 'Life Expectancy (years)'}
)
fig1.update_layout(width=950, height=600, template='plotly_white')
fig1.write_html('day20_charts/github_1_animated_bubble.html')
print('Chart 1 saved: Animated Bubble Chart')

# -------------------------------------------------------
# CHART 2: Animated Choropleth - GDP by Country
# -------------------------------------------------------
fig2 = px.choropleth(
    gapminder,
    locations='iso_alpha',
    color='gdpPercap',
    hover_name='country',
    hover_data={'lifeExp': ':.1f', 'pop': ':,.0f'},
    animation_frame='year',
    color_continuous_scale='Viridis',
    range_color=[0, 50000],
    title='World GDP per Capita (1952-2007) - Animated Choropleth',
    labels={'gdpPercap': 'GDP per Capita (USD)'}
)
fig2.update_layout(
    width=1000, height=520,
    geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth')
)
fig2.write_html('day20_charts/github_2_choropleth.html')
print('Chart 2 saved: Animated Choropleth')

# -------------------------------------------------------
# CHART 3: Bar Chart - Top 10 by Population
# -------------------------------------------------------
top10 = gapminder_2007.nlargest(10, 'pop').sort_values('pop', ascending=True)

fig3 = px.bar(
    top10,
    x='pop', y='country',
    color='continent',
    orientation='h',
    text_auto='.2s',
    hover_data={'gdpPercap': ':,.0f', 'lifeExp': ':.1f'},
    title='Top 10 Countries by Population (2007)',
    labels={'pop': 'Population', 'country': 'Country'}
)
fig3.update_layout(width=800, height=500, template='plotly_white')
fig3.write_html('day20_charts/github_3_bar_top10.html')
print('Chart 3 saved: Bar Chart Top 10')

# -------------------------------------------------------
# CHART 4: Histogram - Life Expectancy, marginal='box'
# -------------------------------------------------------
fig4 = px.histogram(
    gapminder_2007,
    x='lifeExp',
    nbins=25,
    color='continent',
    barmode='overlay',
    opacity=0.65,
    marginal='box',
    hover_data=['country'],
    title='Life Expectancy Distribution by Continent (2007) - marginal=box',
    labels={'lifeExp': 'Life Expectancy (years)'}
)
fig4.update_layout(width=900, height=500, template='plotly_white')
fig4.write_html('day20_charts/github_4_histogram.html')
print('Chart 4 saved: Histogram with marginal box')

print('\nAll Gapminder charts saved in day20_charts/')
print('Open any .html file in browser for full interactivity')

---

# GitHub File 2: graph_objects_dashboard.py - Interactive 2x2 Dashboard

In [ ]:
# ============================================================
# day20_plotly/graph_objects_dashboard.py
# make_subplots 2x2: Scatter, Bar, Box, Pie - Dark Theme
# ============================================================

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import os

os.makedirs('day20_charts', exist_ok=True)
np.random.seed(42)

# --- Fake but realistic sales data ---
months         = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
product_A      = [120, 135, 110, 145, 160, 155, 170, 180, 165, 190, 210, 250]
product_B      = [80,  90,  85,  100, 110, 120, 115, 130, 125, 140, 155, 180]
categories     = ['Electronics', 'Clothing', 'Food', 'Books', 'Sports']
cat_revenue    = [45000, 30000, 25000, 15000, 20000]
region_data    = {'North': [140,160,135,170,185,175,190,200,185,210,230,275],
                  'South': [90, 100,95, 110,120,115,125,135,130,145,160,190]}

# --- Create 2x2 subplot grid ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Monthly Sales: Product A vs B (Line)',
        'Monthly Revenue by Region (Bar)',
        'Sales Distribution (Box with Points)',
        'Revenue Share by Category (Donut)'
    ],
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'box'},     {'type': 'pie'}]
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.08
)

# Top-left: go.Scatter - two lines (Product A and B)
for product, data, color in [('Product A', product_A, '#60a5fa'), ('Product B', product_B, '#f472b6')]:
    fig.add_trace(
        go.Scatter(
            x=months, y=data,
            mode='lines+markers',
            name=product,
            line=dict(color=color, width=2.5),
            marker=dict(size=7),
            hovertemplate=f'{product}<br>Month: %{{x}}<br>Sales: %{{y}} units<extra></extra>'
        ),
        row=1, col=1
    )

# Top-right: go.Bar - stacked bars for two regions
for region, data, color in [('North', region_data['North'], '#34d399'), ('South', region_data['South'], '#fbbf24')]:
    fig.add_trace(
        go.Bar(
            x=months, y=data,
            name=region,
            marker_color=color,
            hovertemplate=f'{region}<br>%{{x}}: %{{y}} units<extra></extra>'
        ),
        row=1, col=2
    )

# Bottom-left: go.Box - one box per product
for product, data, color in [('Product A', product_A, '#60a5fa'), ('Product B', product_B, '#f472b6')]:
    fig.add_trace(
        go.Box(
            y=data,
            name=product,
            marker_color=color,
            boxpoints='all',
            jitter=0.3,
            pointpos=-1.5,
            boxmean=True,        # show mean as dashed line inside box
            hovertemplate='%{y} units<extra></extra>'
        ),
        row=2, col=1
    )

# Bottom-right: go.Pie - donut chart
fig.add_trace(
    go.Pie(
        labels=categories,
        values=cat_revenue,
        hole=0.4,
        name='Revenue Share',
        marker_colors=['#60a5fa','#f472b6','#34d399','#fbbf24','#a78bfa'],
        textinfo='percent+label',
        hovertemplate='%{label}<br>Revenue: $%{value:,}<br>Share: %{percent}<extra></extra>'
    ),
    row=2, col=2
)

# Overall layout - dark professional theme
fig.update_layout(
    title=dict(
        text='Sales Interactive Dashboard',
        font=dict(size=20, color='white'),
        x=0.5
    ),
    width=1200, height=750,
    template='plotly_dark',
    paper_bgcolor='#0f172a',
    plot_bgcolor='#1e293b',
    font=dict(color='white', size=12),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(255,255,255,0.1)',
        bordercolor='rgba(255,255,255,0.2)',
        borderwidth=1
    ),
    barmode='stack'              # stacked bars for the bar chart
)

# Axis labels
fig.update_xaxes(title_text='Month', row=1, col=1, showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.update_yaxes(title_text='Units Sold', row=1, col=1, showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.update_xaxes(title_text='Month', row=1, col=2, showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.update_yaxes(title_text='Units', row=1, col=2)
fig.update_yaxes(title_text='Monthly Sales', row=2, col=1)

# Save as HTML - the portfolio file
fig.write_html(
    'day20_charts/github_dashboard.html',
    include_plotlyjs=True,
    full_html=True
)

fig.show()
print('Interactive dashboard saved: day20_charts/github_dashboard.html')
print('Upload this HTML to GitHub Pages for a live portfolio link')

---

# Complete Reference: Day 20 Cheat Sheet

---

## Plotly Express Quick Reference

```python
import plotly.express as px

# Scatter
px.scatter(df, x=, y=, color=, size=, symbol=,
           hover_name=, hover_data=,
           facet_col=, facet_row=, facet_col_wrap=,
           animation_frame=, animation_group=,
           log_x=, log_y=, range_x=, range_y=,
           size_max=, opacity=, trendline='ols')

# Line
px.line(df, x=, y=, color=, markers=True, line_dash=)

# Bar
px.bar(df, x=, y=, color=,
       barmode='group',   # 'group' | 'stack' | 'overlay' | 'relative'
       orientation='h',   # 'h' horizontal | 'v' vertical
       text_auto=True)    # '.2s' | '.1f' | True

# Histogram
px.histogram(df, x=, nbins=, color=,
             barmode='overlay',  # 'overlay' | 'stack'
             marginal='box',     # 'box' | 'violin' | 'rug'
             opacity=)

# Box
px.box(df, x=, y=, color=,
       points='all',    # 'all' | 'outliers' | False
       notched=True)

# Violin
px.violin(df, x=, y=, color=, box=True, points='all')

# Pie / Donut
px.pie(df, names=, values=, hole=0.4)   # hole=0 pie, hole>0 donut

# Sunburst
px.sunburst(df, path=['col1','col2'], values=, color=)

# Treemap
px.treemap(df, path=['col1','col2'], values=, color=)

# Choropleth Map
px.choropleth(df, locations='iso_alpha', locationmode='ISO-3',
              color=, animation_frame=, color_continuous_scale=)

# Scatter Geo
px.scatter_geo(df, locations='iso_alpha', size=, color=)

# 3D Scatter
px.scatter_3d(df, x=, y=, z=, color=, size=)

# Funnel
px.funnel(df, x=, y=, color=)
```

---

## Graph Objects Quick Reference

```python
import plotly.graph_objects as go

fig = go.Figure()                    # empty figure

# Add traces
fig.add_trace(go.Scatter(
    x=, y=,
    mode='lines+markers',   # 'lines' | 'markers' | 'lines+markers' | 'text'
    name='legend label',
    line=dict(color=, width=, dash='dash'),
    marker=dict(size=, color=, symbol=, opacity=),
    hovertemplate='%{x}: %{y}<extra></extra>'
))

fig.add_trace(go.Bar(x=, y=, marker_color=, text=, textposition=))
fig.add_trace(go.Histogram(x=, nbinsx=, marker_color=))
fig.add_trace(go.Box(y=, boxpoints='all', jitter=, boxmean=True))
fig.add_trace(go.Pie(labels=, values=, hole=))
fig.add_trace(go.Surface(x=XX, y=YY, z=ZZ, colorscale=))

# Customize
fig.update_layout(
    title=, width=, height=, template=,
    xaxis_title=, yaxis_title=,
    showlegend=, legend=dict(...),
    hovermode='x unified',
    barmode='stack'
)
fig.update_traces(marker_color=, opacity=, line_width=)
fig.update_xaxes(title_text=, type='log', row=, col=)
fig.update_yaxes(title_text=, row=, col=)

# Output
fig.show()
fig.write_html('file.html', include_plotlyjs=True)
fig.write_image('file.png', width=, height=, scale=2)  # needs kaleido
```

---

## make_subplots Reference

```python
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Title 1', 'Title 2', 'Title 3', 'Title 4'],
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'box'},     {'type': 'pie'}]
    ],
    # type options: 'scatter', 'bar', 'box', 'pie', 'scene' (3D), 'geo', 'domain'
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# Add trace to specific cell
fig.add_trace(go.Scatter(...), row=1, col=1)
fig.add_trace(go.Bar(...),     row=1, col=2)
fig.add_trace(go.Box(...),     row=2, col=1)
fig.add_trace(go.Pie(...),     row=2, col=2)

# Axis labels for specific subplot
fig.update_xaxes(title_text='Month', row=1, col=1)
fig.update_yaxes(title_text='Value', row=1, col=1)
```

---

## Plotly Templates

```
template='plotly'          # default (light gray background)
template='plotly_white'    # clean white background
template='plotly_dark'     # dark background
template='ggplot2'         # R ggplot2 style
template='seaborn'         # seaborn style
template='simple_white'    # minimal white
template='none'            # no styling
```

---

## Portfolio Tip: GitHub Pages Hosting

```
Steps to host interactive dashboard for free:

1. Create GitHub repository named: your-username.github.io
2. Upload your HTML file (dashboard.html)
3. Enable GitHub Pages in repo Settings -> Pages -> main branch
4. Access at: https://your-username.github.io/dashboard.html
5. Share this link on resume, LinkedIn, and with recruiters

The recruiter can zoom, hover, and interact with your charts directly.
This is 10x more impressive than a static PNG screenshot.
```

---

**Day 20 complete. Plotly Express, Graph Objects, Geographic Maps, Animations, Subplots, HTML Export.**